In [1]:
import pandas as pd
from scipy.stats import ttest_ind

In [2]:
customers = pd.read_csv("/content/customers.csv")
feature_usage = pd.read_csv("/content/feature_usage.csv")
subscription_events=pd.read_csv("/content/subscription_events.csv")
support_tickets=pd.read_csv("/content/support_tickets.csv")


In [3]:
feature_count = feature_usage.groupby("customer_id").agg(
    feature_count=("feature", "nunique")
).reset_index()


In [4]:
df = pd.merge(feature_count, customers, on="customer_id", how="inner")

In [5]:
print("\nColumns in dataset:", df.columns)


Columns in dataset: Index(['customer_id', 'feature_count', 'company_name', 'industry', 'region',
       'plan', 'company_size', 'signup_date', 'mrr', 'health_score',
       'lifecycle_state', 'primary_contact_email', 'num_seats', 'country'],
      dtype='object')


In [6]:
def usage_bucket(x):
    if x <= 3:
        return "Low"
    elif x <= 6:
        return "Medium"
    else:
        return "High"

In [7]:
df["usage_bucket"] = df["feature_count"].apply(usage_bucket)


In [8]:
print("\nAverage MRR by Usage Bucket:")
print(df.groupby("usage_bucket")["mrr"].mean())



Average MRR by Usage Bucket:
usage_bucket
High      626.695944
Medium    118.891349
Name: mrr, dtype: float64


T-TEST

In [11]:
high_usage = df[df["usage_bucket"] == "High"]["mrr"]
medium_usage = df[df["usage_bucket"] == "Medium"]["mrr"]

t_stat, p_value = ttest_ind(high_usage, medium_usage, equal_var=False)

print("\nT-Test Results:")
print("T-statistic:", t_stat)
print("P-value:", p_value)
print("\nInterpretation:")
if p_value < 0.05:
    print("Statistically significant difference ")
    print("Conclusion: Higher feature usage leads to higher revenue")
else:
    print("Not statistically significant ")


T-Test Results:
T-statistic: 50.304800051431435
P-value: 5.327442062722242e-303

Interpretation:
Statistically significant difference 
Conclusion: Higher feature usage leads to higher revenue


compared high vs medium usage customers using a Welch’s t-test and found a statistically significant difference in revenue, confirming that higher feature usage leads to higher monetization.

In [12]:
corr = df["feature_count"].corr(df["mrr"])

print("\nCorrelation between feature usage and MRR:", corr)


Correlation between feature usage and MRR: 0.8223267448037562


There is a strong positive correlation (0.82) between feature usage and MRR, indicating that customers who engage with more features generate significantly higher revenue.

In [15]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(df["usage_bucket"], df["mrr"])

chi2, p, _, _ = chi2_contingency(contingency)

print("\nChi-Square Test:")
print("P-value:", p)

if p < 0.05:
    print("Significant association between usage and revenue ")


Chi-Square Test:
P-value: 2.637136444757306e-07
Significant association between usage and revenue 
